In [1]:
import numpy as np
from numpy.typing import NDArray
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
import lightgbm as lgb

import os
import sys
root_path="/home/iaw/DATA2/AAReact/src"
sys.path.append(root_path)
from util.RegressMetrics import r2_score, mse_score, mae_score, rmse_score
from util.train_tools import build_model, search_parms, print_metric, draw_pred_scatter

import seaborn as sns
from matplotlib import pyplot as plt

from typing import List, Tuple
from rich.table import Table
from rich import print as rp
from rich.progress import track

import pickle

In [6]:
SEED = 1
desc_type = "rdkit_desc"

if desc_type == "rdkit_desc":
    prefix = "/home/iaw/DATA2/AAReact/DataSet/AtropicAcid_416/4_gen_feature/rdkit_3"
    data_x = np.load("{}_data_x.npy".format(prefix))
    data_y = np.load("{}_data_y.npy".format(prefix))
    with open("{}_x_label.pkl".format(prefix), "rb") as f:
        x_label = pickle.load(f)
    with open("{}_data_class.pkl".format(prefix), "rb") as f:
        data_class = pickle.load(f)

X_train, X_test, y_train, y_test,  class_train, class_test = train_test_split(
    data_x,        
    data_y,
    data_class,
    test_size=0.2,
    random_state=SEED, 
)


for model_name in track(["rf", "xgb"]):         # , "lgb"
    print("Infor[iaw]>: start train model: {}".format(model_name))
    best_params = search_parms(model_name, X_train, y_train, SEED)
    print("Infor[iaw]>: {} best params: {}".format(model_name, best_params))
    save_model = build_model(model_name, SEED)
    save_model.set_params(**best_params)
    save_model.fit(X_train, y_train)
    train_pred = save_model.predict(X_train)
    test_pred = save_model.predict(X_test)
    # metrics
    train_r2 = r2_score(y_pred=train_pred, y_true=y_train)
    test_r2 = r2_score(y_pred=test_pred, y_true=y_test)
    train_mse = mse_score(y_pred=train_pred, y_true=y_train)
    test_mse = mse_score(y_pred=test_pred, y_true=y_test)
    train_mae = mae_score(y_pred=train_pred, y_true=y_train)
    test_mae = mae_score(y_pred=test_pred, y_true=y_test)
    train_rmse = rmse_score(y_pred=train_pred, y_true=y_train)
    test_rmse = rmse_score(y_pred=test_pred, y_true=y_test)
    print_metric(["R2", "MSE", "MAE", "RMSE"], [(train_r2, 0, test_r2), (train_mse, 0, test_mse), (train_mae, 0, test_mae), (train_rmse, 0, test_rmse)])
    print("\n")

Output()

Infor[iaw]>: start train model: rf

Infor[iaw]>: rf best params: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 140}

                           Metric                           
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   name    ┃     Train     ┃     Valid     ┃     Test     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│    R2     │    0.8415     │    0.0000     │    0.7457    │
│    MSE    │    0.0422     │    0.0000     │    0.0611    │
│    MAE    │    0.1385     │    0.0000     │    0.1866    │
│   RMSE    │    0.2053     │    0.0000     │    0.2473    │
└───────────┴───────────────┴───────────────┴──────────────┘

Infor[iaw]>: start train model: xgb

Infor[iaw]>: xgb best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 
5, 'n_estimators': 150, 'subsample': 0.8}

                           Metric                           
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   name    ┃     Train     ┃     Valid     ┃     Test     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│    R2     │    0.9704     │    0.0000     │    0.8498    │
│    MSE    │    0.0079     │    0.0000     │    0.0361    │
│    MAE    │    0.0303     │    0.0000     │    0.1140    │
│   RMSE    │    0.0887     │    0.0000     │    0.1900    │
└───────────┴───────────────┴───────────────┴──────────────┘

In [7]:
SEED = 1

prefix = "/home/iaw/DATA2/AAReact/DataSet/AtropicAcid_416/4_gen_feature/soap_3"
data_x = np.load("{}_data_x.npy".format(prefix))
data_y = np.load("{}_data_y.npy".format(prefix))
with open("{}_x_label.pkl".format(prefix), "rb") as f:
    x_label = pickle.load(f)
with open("{}_data_class.pkl".format(prefix), "rb") as f:
    data_class = pickle.load(f)

X_train, X_test, y_train, y_test,  class_train, class_test = train_test_split(
    data_x,        
    data_y,
    data_class,
    test_size=0.2,
    random_state=SEED, 
)


for model_name in track(["rf", "xgb"]):         # , "lgb"
    print("Infor[iaw]>: start train model: {}".format(model_name))
    best_params = search_parms(model_name, X_train, y_train, SEED)
    print("Infor[iaw]>: {} best params: {}".format(model_name, best_params))
    save_model = build_model(model_name, SEED)
    save_model.set_params(**best_params)
    save_model.fit(X_train, y_train)
    train_pred = save_model.predict(X_train)
    test_pred = save_model.predict(X_test)
    # metrics
    train_r2 = r2_score(y_pred=train_pred, y_true=y_train)
    test_r2 = r2_score(y_pred=test_pred, y_true=y_test)
    train_mse = mse_score(y_pred=train_pred, y_true=y_train)
    test_mse = mse_score(y_pred=test_pred, y_true=y_test)
    train_mae = mae_score(y_pred=train_pred, y_true=y_train)
    test_mae = mae_score(y_pred=test_pred, y_true=y_test)
    train_rmse = rmse_score(y_pred=train_pred, y_true=y_train)
    test_rmse = rmse_score(y_pred=test_pred, y_true=y_test)
    print_metric(["R2", "MSE", "MAE", "RMSE"], [(train_r2, 0, test_r2), (train_mse, 0, test_mse), (train_mae, 0, test_mae), (train_rmse, 0, test_rmse)])
    print("\n")

Output()

Infor[iaw]>: start train model: rf

Infor[iaw]>: rf best params: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 120}

                           Metric                           
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   name    ┃     Train     ┃     Valid     ┃     Test     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│    R2     │    0.8678     │    0.0000     │    0.7688    │
│    MSE    │    0.0352     │    0.0000     │    0.0556    │
│    MAE    │    0.1239     │    0.0000     │    0.1764    │
│   RMSE    │    0.1875     │    0.0000     │    0.2358    │
└───────────┴───────────────┴───────────────┴──────────────┘

Infor[iaw]>: start train model: xgb

Infor[iaw]>: xgb best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 
5, 'n_estimators': 51, 'subsample': 1.0}

                           Metric                           
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   name    ┃     Train     ┃     Valid     ┃     Test     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│    R2     │    0.9622     │    0.0000     │    0.8301    │
│    MSE    │    0.0101     │    0.0000     │    0.0408    │
│    MAE    │    0.0445     │    0.0000     │    0.1193    │
│   RMSE    │    0.1003     │    0.0000     │    0.2021    │
└───────────┴───────────────┴───────────────┴──────────────┘

In [8]:
SEED = 1

prefix = "/home/iaw/DATA2/AAReact/DataSet/AtropicAcid_416/4_gen_feature/acsf_3"
data_x = np.load("{}_data_x.npy".format(prefix))
data_y = np.load("{}_data_y.npy".format(prefix))
with open("{}_x_label.pkl".format(prefix), "rb") as f:
    x_label = pickle.load(f)
with open("{}_data_class.pkl".format(prefix), "rb") as f:
    data_class = pickle.load(f)

X_train, X_test, y_train, y_test,  class_train, class_test = train_test_split(
    data_x,        
    data_y,
    data_class,
    test_size=0.2,
    random_state=SEED, 
)


for model_name in track(["rf", "xgb"]):         # , "lgb"
    print("Infor[iaw]>: start train model: {}".format(model_name))
    best_params = search_parms(model_name, X_train, y_train, SEED)
    print("Infor[iaw]>: {} best params: {}".format(model_name, best_params))
    save_model = build_model(model_name, SEED)
    save_model.set_params(**best_params)
    save_model.fit(X_train, y_train)
    train_pred = save_model.predict(X_train)
    test_pred = save_model.predict(X_test)
    # metrics
    train_r2 = r2_score(y_pred=train_pred, y_true=y_train)
    test_r2 = r2_score(y_pred=test_pred, y_true=y_test)
    train_mse = mse_score(y_pred=train_pred, y_true=y_train)
    test_mse = mse_score(y_pred=test_pred, y_true=y_test)
    train_mae = mae_score(y_pred=train_pred, y_true=y_train)
    test_mae = mae_score(y_pred=test_pred, y_true=y_test)
    train_rmse = rmse_score(y_pred=train_pred, y_true=y_train)
    test_rmse = rmse_score(y_pred=test_pred, y_true=y_test)
    print_metric(["R2", "MSE", "MAE", "RMSE"], [(train_r2, 0, test_r2), (train_mse, 0, test_mse), (train_mae, 0, test_mae), (train_rmse, 0, test_rmse)])
    print("\n")

Output()

Infor[iaw]>: start train model: rf

Infor[iaw]>: rf best params: {'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 120}

                           Metric                           
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   name    ┃     Train     ┃     Valid     ┃     Test     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│    R2     │    0.8310     │    0.0000     │    0.7167    │
│    MSE    │    0.0449     │    0.0000     │    0.0681    │
│    MAE    │    0.1451     │    0.0000     │    0.1986    │
│   RMSE    │    0.2120     │    0.0000     │    0.2610    │
└───────────┴───────────────┴───────────────┴──────────────┘

Infor[iaw]>: start train model: xgb

Infor[iaw]>: xgb best params: {'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 20, 'min_child_weight': 
5, 'n_estimators': 120, 'subsample': 1.0}

                           Metric                           
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   name    ┃     Train     ┃     Valid     ┃     Test     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│    R2     │    0.9730     │    0.0000     │    0.8027    │
│    MSE    │    0.0072     │    0.0000     │    0.0474    │
│    MAE    │    0.0270     │    0.0000     │    0.1266    │
│   RMSE    │    0.0848     │    0.0000     │    0.2178    │
└───────────┴───────────────┴───────────────┴──────────────┘

In [ ]:
# rdkit desc_271 3
rf_best_params = {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 140}
xgb_best_params = {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 150, 'subsample': 0.8}

# soap 3 
rf_best_params = {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 120}
xgb_best_params = {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 51, 'subsample': 1.0}

# acsf 3
rf_best_params = {'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 120}
xgb_best_params = {'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 20, 'min_child_weight': 5, 'n_estimators': 120, 'subsample': 1.0}

In [ ]:
from joblib import dump, load

# 需要保存模型权重以及归一化的min和max, 以及用于预测的特征
dump(save_model, "xgb_model_seed_1_split_0-2.pkl")
dump((data_x_min, data_x_max), "data_x_min_max.pkl")